# 03 · Workflows vs Agents: How Much Control to Give the Model

> **Kernel:** `Phase 5 · Concepts` · **Needs Ollama** (local, free). Every
> pattern here makes real model calls. Plain Python otherwise.

Notebook 02 gave you the machinery: nodes, edges, routers, parallel branches,
cycles. This notebook answers the design question that comes before any code:
**for this task, who should decide what happens next: your code, or the model?**

There isn't one right answer. There's a **spectrum**, and six well-known
patterns sit along it. You'll build all six on a real model, see what each one
buys you, and finish with a checklist for choosing.

**You will learn**
- The difference between a **workflow** (code decides the path) and an **agent** (the model decides)
- Six patterns: prompt chaining, routing, parallelisation, orchestrator-workers, evaluator-optimizer, autonomous agent
- What each pattern costs in latency, tokens and predictability
- A decision checklist: start at the simplest point that works

**Prerequisites:** notebooks 01–02 of this section.

> The pattern names follow Anthropic's engineering essay *Building effective
> agents* (Dec 2024), which has become the common vocabulary across frameworks.

## 1 · Why this exists: autonomy is a dial, not a switch

"Should I build an agent?" is the wrong question. The real question is *how
much of the control flow to hand to the model*. Picture a dial:

```
 CODE decides the path ◄─────────────────────────────────────────────► MODEL decides the path

 single     prompt      routing     parallel-   orchestrator-  evaluator-   autonomous
 LLM call   chaining                isation     workers        optimizer    agent
 ────────── ─────────── ─────────── ─────────── ────────────── ──────────── ────────────
            fixed       model       fixed       model plans    model judges model picks
            sequence    picks 1     fan-out     the subtasks   "good enough" every step
                        branch
 ◄──────────────── WORKFLOWS ──────────────────────────────────►◄──────── AGENTS ───────►
 predictable · cheap · testable                          flexible · open-ended · costly
```

- A **workflow** is a graph *you* drew. The model does the work inside the
  nodes, but the path is known in advance or chosen from options you listed.
- An **agent** is a loop where the model chooses the next step, including
  which tools to call and when to stop. The path is discovered at runtime.

Moving right buys flexibility for tasks you can't map out in advance. It costs
predictability, latency, tokens, and ease of testing. The skill is to move
right **only as far as the task requires**.

## 2 · Setup: one small helper around Ollama

`llm()` makes one chat call. With `schema=` it asks Ollama for **structured
output**: the model is constrained to return JSON matching a Pydantic model, so
code can branch on the answer reliably. Every "the model decides" step below
uses that. A router can't act on free text, but it can act on `{"category": "billing"}`.

In [ ]:
import os, time, json
from concurrent.futures import ThreadPoolExecutor
from typing import Literal
import ollama
from pydantic import BaseModel

MODEL = os.getenv('OLLAMA_MODEL', 'qwen3-coder:30b')
CALLS = {'n': 0}

def llm(prompt, system='You are concise and precise.', schema=None, temperature=0.2):
    '''One model call. With `schema` (a Pydantic model), returns a validated object.'''
    CALLS['n'] += 1
    resp = ollama.chat(
        model=MODEL,
        messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}],
        format=schema.model_json_schema() if schema else None,
        options={'temperature': temperature},
    )
    text = resp.message.content
    return schema.model_validate_json(text) if schema else text.strip()

def timed(label, fn):
    CALLS['n'] = 0
    t0 = time.perf_counter()
    out = fn()
    print(f'[{label}] {CALLS["n"]} model calls, {time.perf_counter() - t0:.1f}s')
    return out

try:
    ollama.show(MODEL)
    print('Using', MODEL)
except Exception as err:
    print(f'⚠️  Ollama or {MODEL} not available ({err}). Start Ollama and `ollama pull {MODEL}`,')
    print('   or set OLLAMA_MODEL to a model you have.')

## 3 · Pattern 1: prompt chaining

**Shape:** a fixed sequence of LLM calls, each working on the previous output,
with optional **gates** (plain code checks) between them.

```
START → outline → [gate: exactly 3 points?] → write → END
                          │ no
                          └──► stop early (don't waste the next call)
```

**Use it when** a task splits cleanly into steps and each step is easier than
the whole. You trade a little latency (several calls in sequence) for accuracy,
because each call has one small job.

In [ ]:
class Outline(BaseModel):
    points: list[str]

def prompt_chain(topic):
    outline = llm(f'Give exactly 3 short outline points for a paragraph on: {topic}', schema=Outline)
    print('outline:', outline.points)
    if len(outline.points) != 3:                        # ← the gate is CODE, not a prompt
        return f'Gate failed: expected 3 points, got {len(outline.points)}'
    return llm('Write one tight paragraph (max 80 words) covering, in order:\n- '
               + '\n- '.join(outline.points))

print(timed('chain', lambda: prompt_chain('why checkpoints make AI workflows durable')))

The gate is the important part. It's where code stops a bad intermediate result
from flowing downstream. The model never gets to "decide" to skip it.

## 4 · Pattern 2: routing

**Shape:** one classification call picks **one** of several specialised paths.
It's the `add_router` from notebook 02, with an LLM as the router function.

```
             ┌──► refund handler
START → classify ──► technical handler  → END
             └──► general handler
```

**Use it when** inputs fall into distinct categories that are better handled
separately: different prompts, tools, or even models (a cheap model for easy
questions, a strong one for hard ones). Note that the model only chooses *among
branches you defined*. It can't invent a new one.

In [ ]:
class Route(BaseModel):
    category: Literal['refund', 'technical', 'general']
    reason: str

HANDLERS = {
    'refund':    'You handle refunds. Be empathetic; state the 5-day refund window.',
    'technical': 'You are a support engineer. Give numbered troubleshooting steps.',
    'general':   'You are a friendly support agent. Answer briefly.',
}

def route(ticket):
    r = llm(f'Classify this support ticket:\n{ticket}', schema=Route, temperature=0)
    print(f'→ {r.category:<10} ({r.reason})')
    return llm(ticket, system=HANDLERS[r.category])      # the chosen branch

for ticket in ['I was billed twice for March, I want my money back.',
               'The app freezes when I upload a PDF larger than 10 MB.']:
    print(timed('route', lambda: route(ticket))[:300], '\n')

## 5 · Pattern 3: parallelisation

**Shape:** a fixed **fan-out** of independent calls, then code that joins the
results. There are two flavours:

- **Sectioning:** split one task into independent parts (e.g. check a text for
  tone, facts, and length at the same time).
- **Voting:** ask the *same* question several times and aggregate, trading
  tokens for reliability on judgement calls.

```
          ┌─► check tone   ─┐                    ┌─► vote 1 ─┐
 START ───┼─► check facts  ─┼─► join     START ──┼─► vote 2 ─┼─► majority
          └─► check length ─┘                    └─► vote 3 ─┘
```

**Use it when** the parts are genuinely independent (sectioning) or when one
answer isn't trustworthy enough (voting). Latency ≈ the slowest call, not the sum.

In [ ]:
class Verdict(BaseModel):
    ok: bool
    note: str

DRAFT = ('Our new product is literally the best in the universe and 100% guaranteed to '
         'double your revenue by next week, no matter what.')

LLM_CHECKS = {                                   # judgement calls → the model
    'tone':   'Is the tone professional (no hype)?',
    'claims': 'Are all claims realistic and verifiable?',
}

def sectioning(text):
    ask = lambda q: llm(f'{q}\nText: {text}', schema=Verdict, temperature=0)
    with ThreadPoolExecutor() as pool:                    # fan-out: all checks at once
        results = dict(zip(LLM_CHECKS, pool.map(ask, LLM_CHECKS.values())))
    words = len(text.split())                             # counting → code, not the model
    results['length'] = Verdict(ok=words < 40, note=f'{words} words (limit 40)')
    return {k: ('✅' if v.ok else '❌') + ' ' + v.note for k, v in results.items()}   # fan-in

for k, v in timed('sectioning', lambda: sectioning(DRAFT)).items():
    print(f'  {k:<7}{v}')

Notice that `length` is **not** an LLM check. Models read tokens, not words, so
they are unreliable at counting; asked "is this under 40 words?", a model will
sometimes answer *no* while writing "23 words" in its own explanation. **Anything
code can compute exactly (counts, formats, lookups) should be computed by code.**
Keep the model for judgement.

In [ ]:
class Safe(BaseModel):
    vulnerable: bool

SNIPPET = 'query = "SELECT * FROM users WHERE name = \'" + request.args["name"] + "\'"'

def vote(n=3):
    ask = lambda _: llm(f'Is this code vulnerable to injection?\n{SNIPPET}',
                        schema=Safe, temperature=0.9).vulnerable
    with ThreadPoolExecutor() as pool:
        votes = list(pool.map(ask, range(n)))
    return votes, sum(votes) > n / 2

votes, verdict = timed('voting', vote)
print('votes:', votes, '→ vulnerable' if verdict else '→ safe')

**Note** that the wall-clock time is close to *one* call, not three, because the
branches ran concurrently. But Ollama on one machine may run requests partly in
sequence, so local speed-ups are smaller than with a cloud API.

## 6 · Pattern 4: orchestrator-workers

**Shape:** an **orchestrator** LLM looks at the task and *decides the subtasks*,
then workers handle each one (in parallel), then a synthesiser combines them.

```
                     ┌─► worker(subtask 1) ─┐
START → orchestrator ┼─► worker(subtask 2) ─┼─► synthesise → END
       (plans N)     └─► worker(subtask N) ─┘
                       N is decided at RUNTIME by the model
```

**The difference from parallelisation** is who draws the fan-out. In pattern 3
*you* wrote the three checks. Here the **model** decides how many branches there
are and what they are. That's the first step across the line from "workflow"
toward "agent". (In LangGraph this dynamic fan-out is called `Send`.)

**Use it when** you can't know the subtasks in advance, for example "compare
these products" (how many products?) or "update every file this change touches".

In [ ]:
class Plan(BaseModel):
    subtasks: list[str]

def orchestrator_workers(task):
    plan = llm(f'Break this task into 2-4 independent research subtasks. Task: {task}', schema=Plan)
    print('orchestrator planned:', *[f'\n  • {s}' for s in plan.subtasks])
    with ThreadPoolExecutor() as pool:
        findings = list(pool.map(lambda s: llm(f'In 2 sentences: {s}'), plan.subtasks))
    notes = '\n'.join(f'- {s}: {f}' for s, f in zip(plan.subtasks, findings))
    return llm(f'Task: {task}\nFindings:\n{notes}\n\nWrite a 4-sentence answer.')

print(timed('orchestrator-workers', lambda: orchestrator_workers(
    'Should a small team use SQLite or Postgres to store AI workflow checkpoints?')))

## 7 · Pattern 5: evaluator-optimizer

**Shape:** a generator produces a draft, an evaluator judges it against
criteria, and if it fails, the feedback goes back to the generator. It's a
**cycle**, so it needs a **guard** (max rounds), exactly like notebook 02 §6.

```
START → generate ──► evaluate ──pass──► END
           ▲             │
           └── feedback ─┘ fail (and rounds < MAX)
```

**Use it when** there are clear criteria and iterating measurably helps:
translations, code that must pass tests, text with hard constraints. Here the
**model decides when the result is good enough**, which is more autonomy than
patterns 1–4.

In [ ]:
class Review(BaseModel):
    passed: bool
    feedback: str

def evaluate(tagline):
    '''The evaluator owns the rubric: hard rules in code, taste by the model.'''
    words = tagline.strip('"').split()
    problems = []
    if len(words) > 8:
        problems.append(f'use at most 8 words (you used {len(words)})')
    if 'checkpoint' not in tagline.lower():
        problems.append('mention "checkpoint"')
    if any(w.strip('.,!').lower() in ('ai', 'seamless', 'seamlessly') for w in words):
        problems.append('avoid the words "AI" and "seamless"')
    if '!' in tagline:
        problems.append('no exclamation marks')
    if problems:                                          # fail fast: no model call needed
        return Review(passed=False, feedback='; '.join(problems))
    return llm(f'Is this tagline professional and memorable? Be strict.\nTagline: {tagline}',
               schema=Review, temperature=0)

MAX_ROUNDS = 4

def evaluator_optimizer(brief):
    draft, feedback = None, ''
    for round_ in range(1, MAX_ROUNDS + 1):                       # ← the guard
        prompt = f'Write one product tagline for: {brief}. Reply with the tagline only.'
        if draft:
            prompt += f'\nYour previous tagline: {draft}\nReviewer feedback: {feedback}'
        draft = llm(prompt, temperature=0.8).strip('"')
        review = evaluate(draft)
        print(f'round {round_}: {draft!r}\n         {"PASS" if review.passed else "FAIL"}: {review.feedback}')
        if review.passed:
            break
        feedback = review.feedback
    else:                                                         # loop ended without a PASS
        print(f'⚠️  guard hit after {MAX_ROUNDS} rounds: returning the last draft FLAGGED for human review')
    return draft

timed('evaluator-optimizer', lambda: evaluator_optimizer(
    'a tool that saves AI workflow progress so crashed runs resume'))

The **generator never saw the rubric**. It only learned the rules through the
evaluator's feedback, which is typical: the evaluator can hold criteria that are
long, private, or computed. The evaluator is also a **hybrid**. Cheap exact
rules run in code first, and the model is only asked the subjective question
once those pass. That saves calls and removes counting errors.

Run the cell a few times. Sometimes it passes in round 2, sometimes it hits
`MAX_ROUNDS`. **That second outcome is the guard doing its job**, not a bug.
Without it, a generator that can't satisfy a strict evaluator loops forever and
keeps spending tokens. When the guard fires, don't pretend the result passed:
return it flagged for a human, or fall back to a safe default.

## 8 · Pattern 6: the autonomous agent

**Shape:** the loop from notebook 01. The model picks a tool, sees the result,
picks again, and decides when it's done.

```
START → ┌─► model decides ──tool call──► run tool ─┐
        │                                          │
        └──────────────── observation ◄────────────┘
                     (until the model answers, or a step limit)
```

**Use it when** the steps genuinely can't be predicted: open-ended research,
debugging an unknown failure, navigating an unfamiliar codebase. The model's
judgement is the whole point.

**Its costs** are everything notebook 01 showed: no guaranteed order, a
variable number of steps (so variable cost and latency), and it's hard to test.
In practice, the most robust systems **embed an agent inside a workflow**. The
graph guarantees the must-haves (tests before commit, approval before publish),
and an agent node handles the open-ended part. You'll build exactly that shape
in every framework section's project.

## 9 · The six patterns side by side

| Pattern | Who decides the path | Model calls | Latency | Predictability | Typical use |
|---|---|---|---|---|---|
| Prompt chaining | code (fixed) | fixed, N | sum of steps | high | multi-stage writing, extract → transform |
| Routing | model picks 1 of *your* branches | 1 + branch | 2 steps | high | support triage, model selection |
| Parallelisation | code (fixed fan-out) | fixed, N | ≈ slowest call | high | guardrail checks, voting |
| Orchestrator-workers | model plans the subtasks | variable | plan + slowest + synth | medium | research, multi-file changes |
| Evaluator-optimizer | model judges "good enough" | variable (capped) | rounds × 2 | medium | constrained writing, code + tests |
| Autonomous agent | model decides every step | variable (capped) | unbounded without a cap | low | open-ended exploration |

## 10 · How to choose: a checklist

Work down the list and **stop at the first "yes"**:

1. **Can one well-prompted call (maybe with retrieval or tools) do it?** → Single call. Stop.
2. **Are the steps fixed and known?** → Prompt chaining (add gates between steps).
3. **Do inputs fall into a few known categories?** → Routing.
4. **Are there independent parts, or do you need several opinions?** → Parallelisation.
5. **Are the subtasks unknown until you see the input?** → Orchestrator-workers.
6. **Are there clear pass/fail criteria and does iterating help?** → Evaluator-optimizer.
7. **None of the above: truly open-ended?** → An agent. Put it inside a workflow
   that guarantees the must-haves, with step and token limits.

Patterns **compose**. A router can send hard cases to an orchestrator-workers
branch, whose workers are small agents, followed by an evaluator gate. The
engine from notebook 02, and every framework in this phase, is a way to wire
these patterns together with state and checkpoints.

## 11 · Common mistakes

- **Starting with an agent.** It demos well, but you then spend weeks trying to
  make it predictable. Start with the simplest pattern and move right when the
  task forces you to.
- **Free-text decisions.** A router that returns prose ("I think this is probably
  billing…") needs fragile parsing. Use structured output with a closed set of
  values (`Literal[...]`).
- **Cycles without a guard.** Evaluator-optimizer and agents both need a hard
  maximum on rounds or steps.
- **Parallelising dependent steps.** If step B needs A's output, fan-out gives B
  nothing to work with. Parallelise only independent work.
- **Letting the model re-decide what code already knows.** If the input already
  has a `type` field, route on it with an `if`.

## 12 · Check yourself

<details><summary><b>1.</b> What exactly separates a "workflow" from an "agent"?</summary>

Who decides the path. In a workflow the path is drawn in code, and the model at most picks among branches you defined. In an agent the model chooses each next step (which tool, in what order, when to stop), so the path is discovered at runtime.
</details>

<details><summary><b>2.</b> Parallelisation and orchestrator-workers both fan out. What's the difference, and why does it matter?</summary>

In parallelisation the branches are fixed in code. In orchestrator-workers the model decides how many branches there are and what they do, at runtime. The latter handles tasks whose parts you can't know in advance, but it's less predictable (variable cost, harder to test).
</details>

<details><summary><b>3.</b> Your router sometimes returns "billing-ish" and the code crashes. What's the fix?</summary>

Constrain the output: structured output with a closed set (`Literal['refund','technical','general']`), validated before use. Optionally add a default branch for anything unexpected.
</details>

<details><summary><b>4.</b> Why is voting done with a higher temperature than routing?</summary>

Voting only helps if the samples can differ. Independent "opinions" need some randomness, otherwise you pay for three identical answers. Routing wants the single most likely category every time, so temperature 0.
</details>

<details><summary><b>5.</b> A task: "every night, fetch 5 known RSS feeds, summarise each, email one digest". Which pattern(s), and why not an agent?</summary>

Parallelisation (fixed fan-out over 5 known feeds) followed by a chaining step (combine, then send), plus a checkpoint so a failure doesn't resend or refetch. The steps are fully known, so an agent would add cost and unpredictability for no benefit.
</details>

<details><summary><b>6.</b> What does "embed an agent inside a workflow" buy you?</summary>

The workflow guarantees the non-negotiable structure (order, approvals, checkpoints, limits), while the agent node handles the open-ended part that needs judgement. You get the agent's flexibility where it's needed and the workflow's guarantees everywhere else.
</details>

## Takeaway

Autonomy is a dial. **Workflows** (chaining, routing, parallelisation) keep the
path in code: cheap, fast, testable. **Agentic** patterns (orchestrator-workers,
evaluator-optimizer, autonomous agents) hand path decisions to the model: flexible,
but costlier and less predictable. Start at the simplest point that solves the
task, make every model decision structured, guard every cycle, and embed agents
inside workflows.

**That completes `00_CONCEPTS`.** Every framework section of Phase 5 builds these
same patterns with its own API. Pick any section next; `01_LANGGRAPH` maps most
directly onto the engine you built in notebook 02.